# 04 — Results analysis

Read the prioritizr outputs from `03` (`output_data/iter1_minshortfall30/`) and visualize
them. The notebook **adapts to the run type** recorded in `run_summary.json`:

- **proportion (LP) run** (current prototype) -> one continuous 0-1 allocation surface;
- **binary portfolio run** (later, with Gurobi) -> a suite of near-optimal alternatives.

Views: a **radar/star plot** (axes = 9 continuous features + 1 aggregated EFG axis;
representation per feature), the **allocation / selection-frequency map** over the Y2Y
boundary with PAs, the **solution map(s)**, and a **trade-off table**.

**Kernel:** select **`Python (y2y-geo)`** (the project `.venv`). Ethan runs the cells.

In [ ]:
# ---- Setup --------------------------------------------------------------
import importlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
import geopandas as gpd

import config
importlib.reload(config)
from config import (RESULTS_DIR, RESULTS_SUBDIR, MANIFEST_PATH, HANDOFF_DIR,
                    CORRIDOR_REF, TARGET_CRS, study_area)

RUN_DIR = RESULTS_DIR / RESULTS_SUBDIR
assert RUN_DIR.exists(), f"{RUN_DIR} not found -- run 03 first"
FIG_DIR = config.PROJECT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

manifest = json.loads(Path(MANIFEST_PATH).read_text())
summary = json.loads((RUN_DIR / "run_summary.json").read_text())
rep = pd.read_csv(RUN_DIR / "portfolio_representation.csv")

# eval_feature_representation_summary names the proportion column "relative_held";
# fall back to any "relative*" column if the prioritizr version differs.
REL = "relative_held" if "relative_held" in rep.columns else \
      next(c for c in rep.columns if c.startswith("relative"))

# Adapt to the run type recorded in the summary.
IS_PROP = summary["params"].get("decision_type") == "proportion"
RUN_KIND = "proportion / LP (continuous allocation)" if IS_PROP else "binary portfolio"

print(f"run: {summary['run_tag']} | objective: {summary['objective']} | {RUN_KIND}")
print(f"alternatives: {summary['n_alternatives']} | solve: {summary['solve_seconds']:.1f}s "
      f"| budget {summary['params']['budget_pct']:.0%}")

In [ ]:
# ---- Load portfolio + frequency rasters; validate grid ------------------
portfolio = rioxarray.open_rasterio(RUN_DIR / "portfolio.tif", masked=True)        # (band, y, x)
freq = rioxarray.open_rasterio(RUN_DIR / "selection_frequency.tif", masked=True).squeeze()

g = manifest["grid"]
# Prototype runs may be coarsened (config.PROTOTYPE_AGG_FACTOR), so the output grid can
# differ from the 1 km manifest grid -- require the two output rasters to align, and note
# if the resolution was coarsened.
assert portfolio.rio.width == freq.rio.width and portfolio.rio.height == freq.rio.height, \
    "portfolio and selection_frequency grids differ"
if (portfolio.rio.width, portfolio.rio.height) != (g["width"], g["height"]):
    print(f"note: prototype grid {portfolio.rio.width}x{portfolio.rio.height} "
          f"(coarsened from canonical {g['width']}x{g['height']})")
n_alt = portfolio.sizes["band"]
print(f"portfolio: {n_alt} solution(s) | grid {portfolio.rio.width} x {portfolio.rio.height}, "
      f"{portfolio.rio.crs}")
if IS_PROP:
    print(f"allocation surface: values {float(freq.min()):.2f}..{float(freq.max()):.2f} "
          f"(proportion of each cell selected)")
else:
    print(f"selection frequency: selected in {int(freq.min())}..{int(freq.max())} of {n_alt} alternatives")

In [ ]:
# ---- Radar / star plot: representation per feature ----------------------
cont = [L["name"] for L in manifest["layers"] if L["role"] == "feature_continuous"]
efg  = [L["name"] for L in manifest["layers"] if L["role"] == "feature_efg"]
target = summary["params"]["target_pct"]

def profile(df):
    held = df.set_index("feature")[REL]
    vals = [float(held.get(n, np.nan)) for n in cont]   # one axis per continuous feature
    vals.append(float(held.reindex(efg).mean()))        # aggregated EFG axis (mean held)
    return vals

labels = [n.replace("_", " ") for n in cont] + ["EFG (mean)"]
ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
ang += ang[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for alt, df in rep.groupby("alternative"):
    v = profile(df); v += v[:1]
    ax.plot(ang, v, linewidth=1.4, label=alt)
    ax.fill(ang, v, alpha=0.05)
ax.plot(ang, [target]*len(ang), "--", color="0.4", linewidth=1, label=f"target {target:.0%}")
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(0, max(1.0, float(rep[REL].max())))
head = "Representation by feature (single LP solution)" if n_alt == 1 else \
       f"Representation across {n_alt} near-optimal alternatives"
ax.set_title(head + "\n(relative amount held per feature)", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.30, 1.10), fontsize=7)
fig.savefig(FIG_DIR / "radar_representation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Allocation / selection-frequency map -------------------------------
boundary = gpd.read_file(CORRIDOR_REF).to_crs(TARGET_CRS)
pas = gpd.read_file(config.PA_VECTOR).to_crs(TARGET_CRS)

if IS_PROP:
    cmap, cbar_label = "viridis", "allocation (proportion of cell selected)"
    map_title = "Allocation surface (min-shortfall LP): priority per cell"
    out_png = "priority_allocation.png"
else:
    cmap, cbar_label = "magma", f"times selected (of {n_alt})"
    map_title = "Selection frequency: robust priorities across the near-optimal suite"
    out_png = "priority_selection_frequency.png"

fig, ax = plt.subplots(figsize=(7, 12))
freq.plot.imshow(ax=ax, cmap=cmap, add_colorbar=True,
                 cbar_kwargs=dict(label=cbar_label, shrink=0.5))
pas.boundary.plot(ax=ax, color="cyan", linewidth=0.3, alpha=0.7)
boundary.boundary.plot(ax=ax, color="black", linewidth=0.7)
ax.set_title(map_title); ax.set_aspect("equal"); ax.set_axis_off()
fig.savefig(FIG_DIR / out_png, dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Solution map(s) ----------------------------------------------------
k = min(2, n_alt)
fig, axes = plt.subplots(1, k, figsize=(6.5*k, 11))
axes = np.atleast_1d(axes)
for i in range(k):
    a = axes[i]
    portfolio.isel(band=i).plot.imshow(ax=a, cmap="Greens", add_colorbar=IS_PROP,
                                       cbar_kwargs=dict(label="allocation", shrink=0.4) if IS_PROP else None)
    boundary.boundary.plot(ax=a, color="black", linewidth=0.6)
    panel = "LP solution (allocation)" if (IS_PROP and n_alt == 1) else f"alternative {i+1}"
    a.set_title(panel); a.set_aspect("equal"); a.set_axis_off()
fig.suptitle("Solution: proportion selected per cell" if (IS_PROP and n_alt == 1)
             else "Representative near-optimal alternatives (selected cells)")
fig.savefig(FIG_DIR / "solution_maps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Trade-off table + headline stats -----------------------------------
stats = pd.DataFrame(summary["per_alternative"])
print("Per-solution summary:")
print(stats.to_string(index=False))
if IS_PROP:
    print("  (proportion run: n_selected = summed allocation = effective cells; "
          "n_added_beyond_pa = cells with any allocation outside PAs)")

n_pu = summary["n_planning_units"]; n_locked = summary["n_locked_in"]
print(f"\nregion planning units : {n_pu:,}")
print(f"area budget (cells)   : {summary['budget_cells']:,} ({summary['params']['budget_pct']:.0%})")
print(f"existing PAs locked in: {n_locked:,} ({100*n_locked/n_pu:.1f}% of region)")
if n_alt == 1:
    print(f"area selected         : {stats['pct_region'].iloc[0]:.1f}% of region")
    print(f"added beyond PAs      : {stats['n_added_beyond_pa'].iloc[0]:,.0f} cells")
else:
    print(f"mean area selected    : {stats['pct_region'].mean():.1f}% of region across {n_alt} alternatives")
    print(f"mean added beyond PAs : {stats['n_added_beyond_pa'].mean():,.0f} cells")